# PySpark Declarative Pipelines – Demo Notebook

This notebook demonstrates the **Spark Declarative Pipelines** API introduced in
Apache Spark 4.0.  It walks through:

1. Creating sample user data.
2. Defining a multi-layer pipeline (Bronze → Silver → Gold) using the
   `pyspark.pipelines` decorators.
3. Manually executing each transformation to preview the results.

> **Note:** Running the pipeline via `spark-pipelines run` requires a Spark
> Connect server.  The cells below instead call the underlying transformation
> functions directly so you can explore results interactively.

Reference: https://spark.apache.org/docs/latest/declarative-pipelines-programming-guide.html

## 1. Start a local SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local[*]")
    .appName("declarative-pipelines-demo")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")

## 2. Create sample user data (Bronze layer simulation)

In [ ]:
from pyspark.sql.types import BooleanType, StringType, StructField, StructType

USER_SCHEMA = StructType(
    [
        StructField("id", StringType(), nullable=False),
        StructField("first_name", StringType(), nullable=True),
        StructField("last_name", StringType(), nullable=True),
        StructField("country", StringType(), nullable=True),
        StructField("is_active", BooleanType(), nullable=True),
    ]
)

raw_data = [
    ("1", "Alice", "Smith", "US", True),
    ("2", "Bob", "Jones", "US", False),
    ("3", "Carlo", "Rossi", "IT", True),
    ("4", "Diana", "Prince", "IT", True),
    ("5", "Ethan", "Brown", "UK", False),
    ("6", "Fatima", "Al-Hassan", "UK", True),
]

bronze_df = spark.createDataFrame(raw_data, schema=USER_SCHEMA)
bronze_df.show(truncate=False)
print(f"Total rows: {bronze_df.count()}")

## 3. Silver layer – filter active users & add full_name

In [ ]:
from pipeline.transformations import enrich_with_full_name, filter_active_users

silver_df = enrich_with_full_name(filter_active_users(bronze_df))
silver_df.show(truncate=False)
print(f"Active users: {silver_df.count()}")

## 4. Gold layer – aggregate users by country

In [ ]:
from pipeline.transformations import aggregate_user_count_by_country

gold_df = aggregate_user_count_by_country(silver_df)
gold_df.orderBy("country").show()

## 5. Declarative pipeline definition overview

The `pipeline/demo_pipeline.py` module defines the same logic declaratively
using the `@table` and `@materialized_view` decorators from `pyspark.pipelines`:

```python
from pyspark.pipelines import materialized_view, table

@table(comment="Raw user records ingested from the CSV source.")
def bronze_users():
    spark = SparkSession.getActiveSession()
    return spark.read.option("header", "true").csv("data/users.csv")

@materialized_view(comment="Active users enriched with a full_name column.")
def silver_active_users():
    spark = SparkSession.getActiveSession()
    bronze_df = spark.table("bronze_users")
    return enrich_with_full_name(filter_active_users(bronze_df))

@materialized_view(comment="Aggregated count of active users per country.")
def gold_user_count_by_country():
    spark = SparkSession.getActiveSession()
    silver_df = spark.table("silver_active_users")
    return aggregate_user_count_by_country(silver_df)
```

To run the pipeline end-to-end with a Spark Connect server:

```bash
spark-pipelines run pipeline/demo_pipeline.py
```

## 6. Stop the SparkSession

In [ ]:
spark.stop()
print("SparkSession stopped.")